# Make it so we can produce multiple answers for the numerical optimisation and plot box plots

- Putting a pin in this for now

## Imports

In [295]:
from hashlib import sha256
import time
from tqdm import tqdm
import mmh3
from math import log, pi, sqrt, e
from statistics import NormalDist
from scipy.optimize import minimize, Bounds
import pickle
import numpy as np
import random
from utils import *

## Constants

In [296]:
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%

## Numerical Optimisation

In [297]:
# Optimise 
# add vcost to bound the optimisation
def optimiser(N, p, alpha, factor, t=-1, random_starting_values=False):   # add v_cost if want vanilla cost to be set beforehand

    if t == -1:
        t = round(log(1-p)/log(1-N**(-1/3))) # Calculate t
    mt_max = (2*N)/(t+2)
    mt_target = alpha * mt_max
    if alpha == 1:
        m_0 = N
    else:
        m_0 = round(mt_target/(1-alpha))    # our starting m_0

    
    ## set the bound and target function
    bound = Bounds([1 for i in range(t)], [float('inf') for i in range(t)])
    
    # our target function that needs to be > 0
    # instead of using the target function, we can use the cost function as the objective function
    # need to get the cost of a vanilla rainbow table with the same parameters
    
    # Const * P(vanilla) - P(cherry) > 0
    v_cost, m = vanilla_cost(N, m_0, t)
    v_cost = v_cost * factor
    ineq_cons = {'type': 'ineq', 'fun' : lambda x: v_cost - cost(x, N, m_0)[0]}

    if random_starting_values is False:
        # use 1 as starting values for each Kj
        starting_values = [1 for i in range(t)]
    else: 
        # random starting values for each Kj
        starting_values = np.array(random.sample(range(1, 5000), t))

    
    ## call the optimizer
    # maximise the 
    # 5000000 normally
    res = minimize(lambda Kj: m_t(Kj, N, m_0), starting_values, bounds=bound, constraints=ineq_cons,method = "SLSQP", options={'disp': True, "maxiter": 5000000, "eps": 1, "ftol": 1})
    
    ## res.x is the result of the optimization
    final_cost, m_values = cost(res.x, N, m_0) # calculate the final cost and m values
    return res.x, m_0, final_cost, m_values, starting_values

## Test out different starting values 


In [298]:
kis, m_0, final_cost, m_values, starting_values = optimiser(N, p, 0.5, 5, random_starting_values=True)

Optimization terminated successfully    (Exit mode 0)
            Current function value: -1288.6056169146034
            Iterations: 5
            Function evaluations: 407
            Gradient evaluations: 5


In [299]:
m_values[-1]

1288.6056169146034

In [300]:
for item in kis:
    print(round(item))

286
3755
3241
1627
3453
352
2649
3116
1394
6
2747
3589
3916
3091
3464
768
3122
1327
3968
22
25
1544
3305
3934
2087
1786
1962
1327
3275
295
506
1273
2609
1148
17
185
900
741
10
1511
2620
2237
3208
7
19
3439
2580
1329
715
1409
3902
3686
197
8
608
23
278
11
227
117
3377
2068
3156
8
16
10
22
1386
606
3561
38
2971
1658
3872
3154
2040
8
25
259
4026


In [301]:
(0.5 * ((2 * N)/82))/(1-0.5)

1598.439024390244

In [302]:
with open(f'higher_costs_ftol_1.pickle', 'rb') as f:
    data = pickle.load(f)

other_kis = data[-1][0][0]

for item in other_kis:
    print(round(item))

1
677
806
921
1027
1124
1214
1299
1379
1454
1525
1594
1659
1721
1781
1838
1894
1946
1998
2048
2096
2142
2188
2232
2275
2317
2358
2397
2436
2474
2511
2547
2582
2617
2651
2684
2717
2748
2780
2811
2841
2871
2900
2929
2957
2985
3012
3039
3066
3093
3118
3144
3169
3195
3219
3243
3267
3291
3314
3337
3360
3383
3406
3428
3449
3472
3493
3514
3536
3557
3577
3598
3618
3638
3659
3678
3699
3718
3737
3757
